In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bookstore_ldp_catalog.gold

In [0]:
%sql
select details:flow_progress:data_quality:expectations
from bookstore_ldp_catalog.envent_log_schema.event_log
where event_type= 'flow_progress'

In [0]:
%sql
DESC HISTORY bookstore_ldp_catalog.envent_log_schema.event_log

In [0]:
%sql
select 
  *
from bookstore_ldp_catalog.silver.books_silver

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE bookstore_ldp_catalog.silver.books_sales_silver (
  CONSTRAINT valid_subtotal EXPECT(b.book.subtotal = b.book.quantity * c.price) ON VIOLATION DROP ROW,
  CONSTRAINT valid_total EXPECT(total BETWEEN 0 AND 100000) ON VIOLATION FAIL UPDATE,
  CONSTRAINT valid_date EXPECT(order_timestamp <= current_date() AND year(order_timestamp) >= 2020)
)
AS
SELECT
  *
FROM STREAM(bookstore_ldp_catalog.silver.orders_silver) AS o
CROSS JOIN LATERAL EXPLODE(o.books) b(book)
INNER JOIN bookstore_ldp_catalog.silver.current_books c
  ON b.book.book_id = c.book_id;

In [0]:
%sql
SELECT
  *
FROM bookstore_ldp_catalog.silver.orders_silver o
    --WATERMARK order_timestamp FOR 5 MINUTES AS o
    CROSS JOIN LATERAL EXPLODE(o.books) AS b(book) 

In [0]:
%sql
select * from 
bookstore_ldp_catalog.silver.books_sales_silver